In [1]:
import requests
import psycopg2
from psycopg2.extras import execute_values
from bs4 import BeautifulSoup
import datetime
import time
import schedule
import re

In [2]:
DBcon={'host':'localhost',
        'database':'mydb',
        'user':'ByteMeHarder404',
        'password':'secret123',
        'port':'5432'}

In [22]:
def hackNewsSories():
    print(f"[{datetime.datetime.now()}]Hacker News incoming")
    top_ids = requests.get("https://hacker-news.firebaseio.com/v0/topstories.json").json()[:20]
    stories = []       
    for item_id in top_ids:
        item = requests.get(f"https://hacker-news.firebaseio.com/v0/item/{item_id}.json").json()
        if not item or item.get('type') != 'story':
            continue
        comment_ids = item.get('kids', [])[:5]
        combined_comments = ""
        for c_id in comment_ids:
            c_data = requests.get(f"https://hacker-news.firebaseio.com/v0/item/{c_id}.json").json()
            if c_data and c_data.get('text'):
                clean_comment = re.sub('<[^<]+?>', '', c_data.get('text'))
                combined_comments += f" Comment: {clean_comment} |"
        stories.append((
                    item.get('id'),
                    item.get('title'),
                    item.get('url'),
                    item.get('score'),
                    item.get('by'),
                    datetime.datetime.fromtimestamp(item.get('time')),
                    item.get('text', ''),
                    combined_comments
                ))
        
    conn =psycopg2.connect(**DBcon)
    cur = conn.cursor()
    upsert_query = """INSERT INTO hn_stories (id, title, url, score, by, time, text,top_comments) VALUES %sON CONFLICT (id) DO UPDATE SET score = EXCLUDED.score,top_comments = EXCLUDED.top_comments;"""
    execute_values(cur, upsert_query, stories)
    conn.commit()
    cur.close()
    conn.close()
    print(f"42 HN stories.")

In [19]:
def githtrend():
    print(f"[{datetime.datetime.now()}]GitHub Trending incoming")
    headers = {'User-Agent': 'Mozilla/5.0'}
    url = "https://github.com/trending"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
        
    repos = []
    rows = soup.select('article.Box-row')
        
    for row in rows:
        name_tag = row.select_one('h2 a')
        repo_name = name_tag.get_text(strip=True).replace(' ', '')
        repo_url = "https://github.com" + name_tag['href']
            
        desc_tag = row.select_one('p.col-9')
        description = desc_tag.get_text(strip=True) if desc_tag else ""
        lang_tag = row.select_one('[itemprop="programmingLanguage"]')
        language = lang_tag.get_text(strip=True) if lang_tag else "Unknown"
        meta = row.select_one('div.f6.color-fg-muted.mt-2')
        stars_text = meta.select_one('a[href$="/stargazers"]').get_text(strip=True).replace(',', '')
        stars_total = int(stars_text) if stars_text.isdigit() else 0
            
        today_text = meta.select_one('span.d-inline-block.float-sm-right').get_text(strip=True)
        stars_today = int(re.search(r'\d+', today_text.replace(',', '')).group()) if re.search(r'\d+', today_text) else 0

        repos.append((repo_name, description, language, stars_today, stars_total, repo_url))

    conn = psycopg2.connect(**DBcon)
    cur = conn.cursor()
    upsert_query = """
        INSERT INTO gh_trending (repo_name, description, language, stars_today, stars_total, url)
        VALUES %s
        ON CONFLICT (repo_name) DO UPDATE SET 
            stars_today = EXCLUDED.stars_today,
            stars_total = EXCLUDED.stars_total,
            last_updated = CURRENT_TIMESTAMP;
        """
    execute_values(cur, upsert_query, repos)
    conn.commit()
    cur.close()
    conn.close()
    print(f"{len(repos)} GitHub repos.")

In [23]:
def run_all():
    hackNewsSories()
    githtrend()

In [24]:
schedule.every(10).minutes.do(run_all)
print("Ingestor is on. Press Ctrl+C to stop.")
run_all()
while True:
    schedule.run_pending()
    time.sleep(1)

Ingestor is on. Press Ctrl+C to stop.
[2026-01-12 21:54:56.417264]Hacker News incoming
42 HN stories.
[2026-01-12 21:56:21.034699]GitHub Trending incoming
12 GitHub repos.
[2026-01-12 21:56:23.420392]Hacker News incoming
42 HN stories.
[2026-01-12 21:57:48.895710]GitHub Trending incoming
12 GitHub repos.
[2026-01-12 21:57:51.184464]Hacker News incoming
42 HN stories.
[2026-01-12 21:58:24.746141]GitHub Trending incoming
12 GitHub repos.
[2026-01-12 21:58:27.172015]Hacker News incoming
42 HN stories.
[2026-01-12 21:58:58.439502]GitHub Trending incoming
12 GitHub repos.
[2026-01-12 21:59:01.322704]Hacker News incoming
42 HN stories.
[2026-01-12 22:01:07.354746]GitHub Trending incoming
12 GitHub repos.
[2026-01-12 22:01:10.041622]Hacker News incoming
42 HN stories.
[2026-01-12 22:02:40.355979]GitHub Trending incoming
12 GitHub repos.
[2026-01-12 22:02:43.582507]Hacker News incoming
42 HN stories.
[2026-01-12 22:04:06.139810]GitHub Trending incoming
12 GitHub repos.
[2026-01-12 22:04:57.067

KeyboardInterrupt: 